# Triangle random intersection graphs: explore the results

We now look at the measurements saved by
`03_triangle_simulation.ipynb`. The plot shows how the 3-core fraction
changes with lambda, how much it varies between graphs, and where the
average fraction increases most sharply.

This notebook reads the saved CSV, so you can run it in a fresh kernel
without generating any graphs. Select the project's `.venv` kernel,
review **Analysis settings** below, then run the cells in order.
The final cell displays the figure and saves it, replacing any figure
already at the selected path.

In [ ]:
%matplotlib inline

import csv
from pathlib import Path

import numpy as np
from IPython.display import display
from matplotlib.figure import Figure

# Find the project root whether the kernel starts here or one folder above.
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "notebooks").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
PROJECT_ROOT = NOTEBOOK_DIR.parent

## Analysis settings

Set `data_file` to the CSV produced by the simulation notebook and
choose where to save the plot with `figure_file`. Both paths start
from the project root. To keep an earlier figure, choose a new filename.

The `smoothing_window` sets how many neighbouring lambda points are
averaged when locating the steepest part of the curve. It must be a
positive odd integer. Smoothing only affects this location marker;
the plotted means, medians, and shaded bands use the measurements
directly. The method is explained below.

The graph size and number of trials in the plot title come from the
CSV. To collect measurements for another graph size or lambda range,
change the settings in notebook 03 and run that simulation first.

In [ ]:
config = {
    "data_file": "results/runs/triangle_transition.csv",  # CSV saved by notebook 03.
    "figure_file": "results/figures/triangle_transition.png",  # This file is replaced.
    "smoothing_window": 7,  # Odd number of points averaged to locate the steepest rise.
}
config

## Load the measurements

Each CSV row describes one sampled graph. Keeping the individual
measurements lets us see the variation that an average alone can hide.
The saved settings and seed also allow a particular trial to be
repeated in the simulation notebook.

`load_results` checks that the expected columns are present and that
each `core_fraction` agrees with `core_size / n`. Seeds are read
directly as integers to preserve their exact values.

In [ ]:
def load_results(path: str | Path) -> list[dict]:
    """Read saved trials and check that their counts and fractions agree."""
    fields = ["model", "n", "lambda", "replicate", "seed", "core_size", "core_fraction"]
    rows = []
    with Path(path).open(newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != fields:
            raise ValueError(f"Expected CSV columns: {', '.join(fields)}")
        for row in reader:
            for key in ("n", "replicate", "seed", "core_size"):
                row[key] = int(row[key])
            row["lambda"] = float(row["lambda"])
            row["core_fraction"] = float(row["core_fraction"])
            if (
                row["model"] != "triangle_rig"
                or row["n"] <= 0
                or not 0 <= row["core_size"] <= row["n"]
                or not np.isfinite(row["lambda"])
                or row["lambda"] < 0
                or row["replicate"] < 0
                or row["seed"] < 0
                or not np.isclose(
                    row["core_fraction"],
                    row["core_size"] / row["n"],
                    rtol=1e-9,
                    atol=1e-12,
                )
            ):
                raise ValueError("Invalid measurement or inconsistent core fraction")
            rows.append(row)
    if not rows:
        raise ValueError("The CSV contains no measurements")
    return rows

## Summarize the results and locate the steepest increase

At each lambda, the plot summarizes the core fractions from all the
sampled graphs:

- The **mean** is the average core fraction. The **median** is the
  middle value after sorting, or the average of the two middle values
  when there is an even number of trials.
- The lighter shaded band runs from the smallest to the largest
  observed core fraction.
- The darker band runs from the 25th to the 75th percentile, covering
  the middle half of the results. Its width is the interquartile range.

The bands describe **variation between sampled graphs**, rather than
confidence intervals for the mean. Near a transition, some graphs may
have small cores while others have large ones. Comparing the mean,
median, and bands helps us notice this mixture of outcomes.

To mark where the average core fraction rises most sharply, we first
smooth the means with a moving average. We estimate the slope of this
smoothed curve and select the lambda with the largest positive slope,
excluding the first and last grid points. Near the ends, the moving
average repeats the endpoint value; for a short grid, its window is
reduced to fit. There is no marker when fewer than three points are
available, the means are flat, or no interior slope is positive.

This marker is a practical description of a **finite simulation**.
Its location depends on the graph size, chosen lambda values, number
of trials, and smoothing window. It does not establish the critical
parameter as $n \to \infty$. That requires comparing graph sizes and,
later, relating the observations to a theoretical prediction.

In [ ]:
def estimate_transition(lambdas, means, window: int = 7) -> float | None:
    """Locate the steepest rise in the smoothed means, excluding endpoints.

    Return None if the curve is too short, flat, or has no positive
    interior slope. The location describes these finite measurements.
    """
    if window < 1 or window % 2 == 0:
        raise ValueError("The smoothing window must be a positive odd integer")
    if len(lambdas) < 3 or np.allclose(
        means, means[0], rtol=0, atol=np.finfo(float).eps
    ):
        return None

    # Keep the window odd and small enough to fit the available grid.
    # Repeating endpoint values keeps one smoothed value per grid point.
    window = min(window, len(lambdas) if len(lambdas) % 2 else len(lambdas) - 1)
    padded = np.pad(means, window // 2, mode="edge")
    smoothed = np.convolve(padded, np.ones(window) / window, mode="valid")
    slopes = np.gradient(smoothed, lambdas)
    index = int(np.argmax(slopes[1:-1])) + 1
    return float(lambdas[index]) if slopes[index] > 0 else None


def plot_results(rows: list[dict], smoothing_window: int = 7):
    """Plot core fractions for one graph size and mark the steepest rise.

    Group the trials by lambda regardless of their input order.
    Lines show the mean and median; bands show the full observed range
    and the middle half of the results. Only the location marker uses
    smoothing. Return both the figure and that estimated location.
    """
    if not rows or len({row["n"] for row in rows}) != 1:
        raise ValueError("Provide measurements for exactly one graph size")
    grouped = {}
    for row in rows:
        grouped.setdefault(row["lambda"], []).append(row["core_fraction"])
    lambdas = np.array(sorted(grouped))
    samples = [grouped[lam] for lam in lambdas]
    means = np.array([np.mean(sample) for sample in samples])
    medians = [np.median(sample) for sample in samples]
    minima = [min(sample) for sample in samples]
    maxima = [max(sample) for sample in samples]
    q25 = [np.percentile(sample, 25) for sample in samples]
    q75 = [np.percentile(sample, 75) for sample in samples]
    estimate = estimate_transition(lambdas, means, smoothing_window)

    figure = Figure(figsize=(10, 6), layout="constrained")
    axis = figure.subplots()
    axis.fill_between(
        lambdas,
        minima,
        maxima,
        color="tab:blue",
        alpha=0.12,
        label="Min–max range",
    )
    axis.fill_between(
        lambdas,
        q25,
        q75,
        color="tab:blue",
        alpha=0.3,
        label="25th–75th percentile",
    )
    axis.plot(lambdas, means, color="tab:blue", linewidth=2, label="Mean")
    axis.plot(
        lambdas,
        medians,
        color="tab:orange",
        linestyle="--",
        linewidth=1.5,
        label="Median",
    )
    if estimate is not None:
        axis.axvline(
            estimate,
            color="tab:red",
            linestyle="--",
            linewidth=2,
            label=f"Finite-size slope estimate: {estimate:.3f}",
        )
    counts = [len(sample) for sample in samples]
    repeats = (
        str(counts[0]) if min(counts) == max(counts) else f"{min(counts)}–{max(counts)}"
    )
    axis.set_title(
        f"3-core transition in triangle RIGs (n={rows[0]['n']}, {repeats} trials per λ)"
    )
    axis.set_xlabel(r"$\lambda$")
    axis.set_ylabel(r"3-core fraction $|\mathrm{Core}_3(G)|/n$")
    if len(lambdas) > 1:
        axis.set_xlim(lambdas[0], lambdas[-1])
    axis.set_ylim(0, 1)
    axis.grid(True, alpha=0.3)
    axis.legend()
    return figure, estimate

In [ ]:
data_path = PROJECT_ROOT / config["data_file"]
if not data_path.is_file():
    raise FileNotFoundError(
        f"No saved measurements at {data_path}. "
        "Run make simulate from the project root, or run notebook 03 first."
    )
rows = load_results(data_path)
print(
    f"Loaded {len(rows)} realizations at "
    f"{len({row['lambda'] for row in rows})} lambda values from {data_path}."
)

## Draw the plot and consider what it shows

The horizontal axis is the Poisson membership parameter $\lambda$.
The vertical axis is the fraction of all individuals in the 3-core,
$|\operatorname{Core}_3(G)|/n$: zero means the core is empty, and one
means every individual survives pruning.

Look at the average and the variation together. A sharp change in the
mean can accompany very different outcomes among individual graphs.
If the slope marker is close to either end of the lambda range, try
a wider range before interpreting its location. The cell below saves
the figure and displays it here.

In [ ]:
figure, transition_estimate = plot_results(rows, config["smoothing_window"])
figure_path = PROJECT_ROOT / config["figure_file"]
figure_path.parent.mkdir(parents=True, exist_ok=True)
figure.savefig(figure_path, dpi=160)
display(figure)

print(f"Saved figure to {figure_path}.")
if transition_estimate is None:
    print("No finite-size slope estimate for this short or flat curve.")
else:
    print(f"Finite-size maximum-slope location: {transition_estimate:.4f}")